<b>Transform Refunds Data

1. Extract specific portion of the string from refund_reason using split function  
2. Extract specific portion of the string from refund_reason using regexp_extract function  
3. Extract date and time from the refund_timestamp  
4. Write transformed data to the Silver schema 

In [0]:
df_refunds = spark.table('gizmobox.bronze.py_refunds')
display(df_refunds)

<b>1. Extract specific portion of the string from refund_reason using split function

[Documentation for split Function](link)

In [0]:
from pyspark.sql import functions as f

df_split_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            "refund_date",
            "refund_amount",
            f.split("refund_reason", ":") [0].alias("refund_reason"),
            f.split("refund_reason", ":") [1].alias("refund_source")
    )
)

display(df_split_refunds)


<b>2. Extract specific portion of the string from refund_reason using regexp_extract function
<br>
[Documentation for regexp_extract Function](link)

In [0]:
df_transfromed_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            f.date_format("refund_date", "yyyy-MM-dd").alias("refund_date"),
            f.date_format("refund_date", "HH:mm:ss").alias("refund_time"),
            "refund_amount",
            f.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
            f.regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("refund_source")
        )
)
display(df_transfromed_refunds)

<b>3.Write transformed data to the Silver schema 

In [0]:
df_transfromed_refunds.writeTo("gizmobox.silver.py_refunds").createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_refunds